# 7C. Campaign Recommendations and Evaluation Plan

## Purpose and boundaries
Turn Part B associations into six documented campaign hypotheses, one per segment. This is a design stage, not another mining stage. Campaigns are proposed, not approved or executed.

The project asks for segment-specific campaigns and feasible mechanics. Not every segment needs a strong pair-rule claim. Weak findings and commercially awkward pairs remain visible rather than being hidden.

Selected hypotheses are post-hoc and informed by the already-inspected discovery and holdout results. No untouched test set or causal uplift is claimed. Customer segment names are descriptive labels, not evidence of lifestyle, family status, price elasticity, or future preferences.

### Prerequisites
- Run Part B and retain its original customer split.
- Put `campaign_recommendation_functions.py` in `utils/`.
- This notebook reads `pair_rules_discovery_with_checks.csv` from the Part B table directory.
- No new external package beyond pandas and notebook display utilities is required.

### Revised analytical review
Adds explicit campaign decisions, segment-fit limitations, and report-ready interpretation. Part B rules and the six proposals are unchanged. An evidence flag, an analytical recommendation, commercial approval, and launch status are four separate concepts.

Keep the existing `utils/campaign_recommendation_functions.py` and add the supplied `utils/campaign_review_functions.py`. Human annotations are provisional judgments based on the inspected evidence, not fitted scores or a new independent validation.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from utils.campaign_recommendation_functions import (
    campaign_specs, load_rule_evidence, build_campaign_plan,
    experiment_checklist, write_report,
)

part_b_dir = project_root / "reports" / "tables" / "association_rules" / "part_b"
output_dir = project_root / "reports" / "tables" / "association_rules" / "part_c"
output_dir.mkdir(parents=True, exist_ok=True)
rule_path = part_b_dir / "pair_rules_discovery_with_checks.csv"
assert rule_path.exists(), f"Run Part B first: {rule_path}"
rules = load_rule_evidence(rule_path)
print(f"Loaded {len(rules):,} directed rules from Part B.")
from utils.campaign_review_functions import attach_review, write_reviewed_report

Loaded 2,803 directed rules from Part B.


## 1. Proposed campaign hypotheses
The six starting proposals deliberately avoid restricted-product promotions and do not automatically adopt the largest lift. Exact product names and categories must match the accepted mapping. Category associations do not establish that every SKU combination makes sense.

Selection is human-reviewed and exploratory. Higher confidence in one direction describes a conditional basket proportion, not purchase order or the best advertising direction. No new numerical threshold is introduced to make these proposals pass.

In [2]:
specs = campaign_specs()
display(specs)
plan = build_campaign_plan(rules, specs)
assert set(plan["cluster"]) == set(range(6))
assert plan["campaign_id"].is_unique
assert not plan["launch_approved"].any()
plan = attach_review(plan)
assert not plan["launch_approved"].any()
assert plan["review_status"].eq("pending_commercial_review").all()

,cluster,segment_name,level,antecedent,consequent,campaign_name,proposed_mechanic,commercial_caution
0,0,Established Multi-Category Shoppers,category,fish,nonalcohol_drinks,Meal companion,Show a relevant non-alcoholic drink alongside ...,Category-level pairing; exact SKU compatibilit...
1,1,Electronics and Gaming Shoppers,item,frozen smoothie,cottage cheese,Food pairing pilot,Test a small complementary-food recommendation...,Food pairing needs merchandising review; not a...
2,2,"Low-Spending, Promotion-Oriented Shoppers",item,fresh tuna,barbecue sauce,Meal preparation suggestion,Test a recipe-compatible sauce recommendation ...,Check exact product suitability and margin bef...
3,3,Promotion-Heavy Regular Shoppers,category,fish,hygiene,Cross-aisle essentials,Test a separate household-essentials reminder ...,Cross-category association does not demonstrat...
4,4,High-Spending Multi-Category Shoppers,item,carrots,tomatoes,Vegetable pairing,Test a recipe-led tomato suggestion alongside ...,Do not infer premium preferences or willingnes...
5,5,Vegetable-Focused Shoppers,item,strong cheese,frozen vegetables,Vegetable meal companion,Test a recipe-compatible frozen-vegetable sugg...,Small supporting holdout count; prioritize a l...


## 2. Attach evidence without changing the mining
Keep counts, confidence, lift, and comparisons with other discovery segments side by side. A minimum descriptive holdout pass is not approval. Exact numbers are read from your export, never copied from earlier notebook output.

Cluster 5 merits special caution because Part B showed weaker overall persistence. A specific plausible pair can still be documented as a small pilot hypothesis, not a strong segment-wide conclusion.

In [3]:
evidence_columns = [
    "campaign_id", "cluster", "campaign_name", "level", "antecedent", "consequent",
    "joint_count", "joint_customers", "confidence", "lift",
    "confidence_difference_vs_rest", "holdout_joint_count",
    "holdout_confidence", "holdout_lift", "evidence_status", "analytical_decision", "holdout_review_gate", "review_status",
]
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(plan[evidence_columns].round(3))

deferred = plan.loc[~plan["holdout_minimum_evidence_check"], evidence_columns]
if not deferred.empty:
    print("Do not launch these proposals on the present rule evidence:")
    display(deferred)

,campaign_id,cluster,campaign_name,level,antecedent,consequent,joint_count,joint_customers,confidence,lift,confidence_difference_vs_rest,holdout_joint_count,holdout_confidence,holdout_lift,evidence_status,analytical_decision,holdout_review_gate,review_status
0,C0-01,0,Meal companion,category,fish,nonalcohol_drinks,4710,2572,0.525,1.188,-0.025,1224,0.526,1.155,minimum descriptive evidence met,retain_modest_hypothesis,minimum_descriptive_evidence_met,pending_commercial_review
1,C1-01,1,Food pairing pilot,item,frozen smoothie,cottage cheese,68,48,0.209,4.669,0.025,15,0.170,3.481,minimum descriptive evidence met,exploratory_alternative,minimum_descriptive_evidence_met,pending_commercial_review
2,C2-01,2,Meal preparation suggestion,item,fresh tuna,barbecue sauce,122,89,0.249,4.457,0.073,20,0.187,4.324,minimum descriptive evidence met,retain_for_compatibility_review,minimum_descriptive_evidence_met,pending_commercial_review
3,C3-01,3,Cross-aisle essentials,category,fish,hygiene,2483,1378,0.649,1.130,0.249,588,0.624,1.098,minimum descriptive evidence met,deprioritize,minimum_descriptive_evidence_met,pending_commercial_review
4,C4-01,4,Vegetable pairing,item,carrots,tomatoes,163,73,0.351,6.485,-0.260,37,0.411,7.949,minimum descriptive evidence met,retain_recipe_hypothesis,minimum_descriptive_evidence_met,pending_commercial_review
5,C5-01,5,Vegetable meal companion,item,strong cheese,frozen vegetables,56,55,0.299,1.322,0.263,15,0.319,1.549,minimum descriptive evidence met,limited_exploratory_review,minimum_descriptive_evidence_met,pending_commercial_review


## 3. Campaign prioritization and segment fit

Four separate questions are shown: does the rule meet the descriptive evidence gate; is the idea analytically worth pursuing; has commercial review approved it; has launch been authorized? A positive answer to the first never implies the others.

The segment-fit rationale distinguishes consistency with a descriptive label from independently verified needs. This notebook does not reload and audit Notebook 6 profiles. Unverified links stay explicit; it does not invent a lifestyle, family status, willingness to pay, or price elasticity.

The decision labels are qualitative, not a learned score or a guaranteed priority ordering. They must be reconsidered if evidence or proposal definitions change. A failed holdout gate requires deferral regardless of the retained qualitative label.

In [4]:
review_view = ["campaign_id", "analytical_decision", "holdout_review_gate",
    "segment_fit_status", "segment_fit_rationale", "decision_rationale", "mechanic_rationale",
    "segment_specificity_note", "review_status", "launch_approved"]
with pd.option_context("display.max_columns", None, "display.max_colwidth", 110):
    display(plan[review_view])

# Descriptive effect sizes: no significance or uplift interpretation.
comparison = plan[["campaign_id", "joint_count", "holdout_joint_count",
    "confidence", "holdout_confidence", "lift", "holdout_lift",
    "discovery_confidence_difference_pp"]].copy()
display(comparison.round(3))

,campaign_id,analytical_decision,holdout_review_gate,segment_fit_status,segment_fit_rationale,decision_rationale,mechanic_rationale,segment_specificity_note,review_status,launch_approved
0,C0-01,retain_modest_hypothesis,minimum_descriptive_evidence_met,not_established,Broad shopping label alone does not establish a fish/drink need; association is the direct evidence.,"Review exact meal-compatible drinks. Modest association, not exclusive segment opportunity.",No-price-change meal recommendation; no demonstrated benefit from adding a discount.,Discovery confidence is not higher than rest; no targeting superiority established,pending_commercial_review,False
1,C1-01,exploratory_alternative,minimum_descriptive_evidence_met,not_established,Food pairing does not directly support the electronics/gaming segment interpretation.,Needs stronger profile rationale and product compatibility review; limited holdout joint count.,Small recommendation pilot only if merchandising agrees; no gaming or family persona claims.,"Discovery confidence is higher than rest; descriptive difference, not causal targeting advantage",pending_commercial_review,False
2,C2-01,retain_for_compatibility_review,minimum_descriptive_evidence_met,not_established,Promotion-oriented label does not establish price elasticity; food need is not independently profiled here.,Check that the exact sauce and tuna form a credible recipe; limited holdout joint count.,Recipe-led recommendation first. Define coupon terms only after reviewing economics.,"Discovery confidence is higher than rest; descriptive difference, not causal targeting advantage",pending_commercial_review,False
3,C3-01,deprioritize,minimum_descriptive_evidence_met,not_established,Promotion-heavy label does not justify a fish/hygiene combination or a blanket discount.,Cross-aisle observation with weak commercial complementarity; do not prioritize a pilot yet.,Hold as descriptive evidence pending a defensible household-essentials use case.,"Discovery confidence is higher than rest; descriptive difference, not causal targeting advantage",pending_commercial_review,False
4,C4-01,retain_recipe_hypothesis,minimum_descriptive_evidence_met,not_established,High spending does not imply premium preferences; pairing alone does not prove superior targeting.,"Recipe logic is plausible, but outside-segment confidence comparison limits targeting claims.",Recipe-led vegetable recommendation without an assumed willingness to pay or automatic discount.,Discovery confidence is not higher than rest; no targeting superiority established,pending_commercial_review,False
5,C5-01,limited_exploratory_review,minimum_descriptive_evidence_met,profile_consistent_not_verified,Vegetable consequent is consistent with the descriptive segment label; no independent profile audit is per...,Potential meal pairing; small holdout evidence and weaker overall segment persistence require caution.,"Limited recipe-led hypothesis, not broad rollout or a claim that every vegetable shopper wants cheese.","Discovery confidence is higher than rest; descriptive difference, not causal targeting advantage",pending_commercial_review,False


,campaign_id,joint_count,holdout_joint_count,confidence,holdout_confidence,lift,holdout_lift,discovery_confidence_difference_pp
0,C0-01,4710,1224,0.525,0.526,1.188,1.155,-2.487
1,C1-01,68,15,0.209,0.170,4.669,3.481,2.493
2,C2-01,122,20,0.249,0.187,4.457,4.324,7.339
3,C3-01,2483,588,0.649,0.624,1.130,1.098,24.890
4,C4-01,163,37,0.351,0.411,6.485,7.949,-25.968
5,C5-01,56,15,0.299,0.319,1.322,1.549,26.291


## 4. Rejected or deferred ideas
Document why attractive-looking associations were not selected. This is not a claim that the remaining proposals are optimal.

- Cologne with oil: commercially unclear, even when lift exceeds one.
- Videogames with baby food or vegetables: do not invent household or gamer personas from co-occurrence.
- Mineral water with cauliflower and low-fat yogurt with green beans: Part B showed holdout weakening; inspect rather than sell as robust cross-sells.
- Blanket discounts: no cost, margin, or incremental-response evidence supports a discount amount.

A campaign may use profile evidence rather than a pair rule in a later revision, but it must explicitly state that evidence basis. Do not silently replace a missing rule or retune thresholds.

In [5]:
review_log = pd.DataFrame([
    {"idea": "Cologne + oil", "decision": "descriptive_only", "reason": "No clear complementary offer logic"},
    {"idea": "Videogames + food", "decision": "descriptive_only", "reason": "Avoid unsupported lifestyle or family inference"},
    {"idea": "Mineral water -> cauliflower", "decision": "defer", "reason": "Positive association did not persist in inspected holdout"},
    {"idea": "Low fat yogurt -> green beans", "decision": "defer", "reason": "Positive association did not persist in inspected holdout"},
    {"idea": "Blanket or automatic percentage discount", "decision": "not_specified", "reason": "Commercial economics unavailable"},
])
display(review_log)

,idea,decision,reason
0,Cologne + oil,descriptive_only,No clear complementary offer logic
1,Videogames + food,descriptive_only,Avoid unsupported lifestyle or family inference
2,Mineral water -> cauliflower,defer,Positive association did not persist in inspec...
3,Low fat yogurt -> green beans,defer,Positive association did not persist in inspec...
4,Blanket or automatic percentage discount,not_specified,Commercial economics unavailable


## 5. Commercial review and audience scope
All proposals remain pending. Edit a separate review record after merchandising assessment; do not rewrite evidence to match a preferred campaign.

Future eligibility is an assigned segment plus a qualifying live basket and operational eligibility. Historical sampled-basket counts do not estimate live reach or shopping frequency. This notebook does not generate contact lists, allocate customers, or send communications.

Check product compatibility, price and margin, stock, consent, channel access, contact limits, and relevant eligibility policies. Customers with unresolved cluster assignments are outside this draft campaign scope; the final all-customer cluster deliverable must still be completed separately.

In [6]:
review_columns = ["campaign_id", "campaign_name", "proposed_mechanic", "commercial_caution",
                  "audience_definition", "analytical_decision", "holdout_review_gate", "discount_amount", "review_status", "launch_approved"]
display(plan[review_columns])

,campaign_id,campaign_name,proposed_mechanic,commercial_caution,audience_definition,analytical_decision,holdout_review_gate,discount_amount,review_status,launch_approved
0,C0-01,Meal companion,Show a relevant non-alcoholic drink alongside ...,Category-level pairing; exact SKU compatibilit...,Assigned segment AND qualifying antecedent in ...,retain_modest_hypothesis,minimum_descriptive_evidence_met,<NA>,pending_commercial_review,False
1,C1-01,Food pairing pilot,Test a small complementary-food recommendation...,Food pairing needs merchandising review; not a...,Assigned segment AND qualifying antecedent in ...,exploratory_alternative,minimum_descriptive_evidence_met,<NA>,pending_commercial_review,False
2,C2-01,Meal preparation suggestion,Test a recipe-compatible sauce recommendation ...,Check exact product suitability and margin bef...,Assigned segment AND qualifying antecedent in ...,retain_for_compatibility_review,minimum_descriptive_evidence_met,<NA>,pending_commercial_review,False
3,C3-01,Cross-aisle essentials,Test a separate household-essentials reminder ...,Cross-category association does not demonstrat...,Assigned segment AND qualifying antecedent in ...,deprioritize,minimum_descriptive_evidence_met,<NA>,pending_commercial_review,False
4,C4-01,Vegetable pairing,Test a recipe-led tomato suggestion alongside ...,Do not infer premium preferences or willingnes...,Assigned segment AND qualifying antecedent in ...,retain_recipe_hypothesis,minimum_descriptive_evidence_met,<NA>,pending_commercial_review,False
5,C5-01,Vegetable meal companion,Test a recipe-compatible frozen-vegetable sugg...,Small supporting holdout count; prioritize a l...,Assigned segment AND qualifying antecedent in ...,limited_exploratory_review,minimum_descriptive_evidence_met,<NA>,pending_commercial_review,False


## 6. Proposed prospective evaluation
This is a proposed design, not a result or guarantee.

1. Define the eligible population and qualifying basket trigger before launch.
2. Randomize eligible customers, retaining the same treatment across repeat baskets. Log exposure and assignment.
3. Compare a recommendation experience with business as usual. If a discount is later proposed, define it as a separate treatment rather than changing the campaign mid-test.
4. Predeclare the time window, contribution-margin definition, minimum worthwhile effect, sample-size calculation, exclusions, and analysis method. Include customers who do not convert in the assigned-group analysis.
5. Use incremental contribution margin per randomized eligible customer as the proposed primary metric. Secondary outcomes include consequent purchase rate and basket value. Record complaints, opt-outs, stockouts, returns, and margin dilution as guardrails.
6. Account for repeat observations at customer level. Define how the six campaign comparisons and any stopping decisions will be handled before looking at outcomes.

No numeric sample size, uplift forecast, or ROI can be justified from the pair-rule table alone. Baseline outcomes, variability, operational costs, and a business-relevant effect size are still needed.

Only ideas retained after analytical and commercial review should advance to a pilot. A checklist row is a design record, not a test commitment. C3-01 is deprioritized; C1-01 remains an exploratory alternative with unresolved segment fit. Margin data and live eligibility are not supplied by sampled baskets.

In [7]:
test_plan = experiment_checklist(plan)
display(test_plan)
assert test_plan["sample_size_per_arm"].isna().all()
assert test_plan["test_status"].eq("design_only_not_launched").all()
test_plan = test_plan.merge(
    plan[["campaign_id", "analytical_decision", "holdout_review_gate"]],
    on="campaign_id", validate="one_to_one",
)
display(test_plan[["campaign_id", "analytical_decision", "holdout_review_gate", "test_status"]])

,campaign_id,cluster,campaign_name,randomization_unit,control,primary_metric,secondary_metrics,guardrails,launch_requirements,owner,baseline_rate,baseline_margin_variance,minimum_detectable_effect,sample_size_per_arm,test_window,analysis_plan,approval_reference,test_status
0,C0-01,0,Meal companion,customer; fixed assignment across repeat baskets,Business-as-usual experience without the added...,Incremental contribution margin per randomized...,Consequent purchase rate; basket value; redemp...,"Complaints/opt-outs, returns, stock availabili...","Merchandising approval, stock, consent, margin...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,design_only_not_launched
1,C1-01,1,Food pairing pilot,customer; fixed assignment across repeat baskets,Business-as-usual experience without the added...,Incremental contribution margin per randomized...,Consequent purchase rate; basket value; redemp...,"Complaints/opt-outs, returns, stock availabili...","Merchandising approval, stock, consent, margin...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,design_only_not_launched
2,C2-01,2,Meal preparation suggestion,customer; fixed assignment across repeat baskets,Business-as-usual experience without the added...,Incremental contribution margin per randomized...,Consequent purchase rate; basket value; redemp...,"Complaints/opt-outs, returns, stock availabili...","Merchandising approval, stock, consent, margin...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,design_only_not_launched
3,C3-01,3,Cross-aisle essentials,customer; fixed assignment across repeat baskets,Business-as-usual experience without the added...,Incremental contribution margin per randomized...,Consequent purchase rate; basket value; redemp...,"Complaints/opt-outs, returns, stock availabili...","Merchandising approval, stock, consent, margin...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,design_only_not_launched
4,C4-01,4,Vegetable pairing,customer; fixed assignment across repeat baskets,Business-as-usual experience without the added...,Incremental contribution margin per randomized...,Consequent purchase rate; basket value; redemp...,"Complaints/opt-outs, returns, stock availabili...","Merchandising approval, stock, consent, margin...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,design_only_not_launched
5,C5-01,5,Vegetable meal companion,customer; fixed assignment across repeat baskets,Business-as-usual experience without the added...,Incremental contribution margin per randomized...,Consequent purchase rate; basket value; redemp...,"Complaints/opt-outs, returns, stock availabili...","Merchandising approval, stock, consent, margin...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,design_only_not_launched


,campaign_id,analytical_decision,holdout_review_gate,test_status
0,C0-01,retain_modest_hypothesis,minimum_descriptive_evidence_met,design_only_not_launched
1,C1-01,exploratory_alternative,minimum_descriptive_evidence_met,design_only_not_launched
2,C2-01,retain_for_compatibility_review,minimum_descriptive_evidence_met,design_only_not_launched
3,C3-01,deprioritize,minimum_descriptive_evidence_met,design_only_not_launched
4,C4-01,retain_recipe_hypothesis,minimum_descriptive_evidence_met,design_only_not_launched
5,C5-01,limited_exploratory_review,minimum_descriptive_evidence_met,design_only_not_launched


## 7. Export proposal artifacts
Outputs are draft recommendations, evidence snapshots, a review log, an experiment checklist, and a report-ready narrative. None authorizes a launch. Rerunning overwrites the generated drafts: keep completed human approvals in a separately versioned review file.

In [8]:
plan.to_csv(output_dir / "campaign_proposals_with_evidence.csv", index=False)
review_log.to_csv(output_dir / "campaign_rejection_log.csv", index=False)
test_plan.to_csv(output_dir / "campaign_experiment_checklist.csv", index=False)
write_reviewed_report(plan, output_dir / "campaign_recommendations.md")
settings = {
    "input_rule_export": str(rule_path.relative_to(project_root)),
    "selection": "human-designed post-hoc exploratory hypotheses informed by inspected Part B evidence",
    "new_mining": False,
    "launch_approved": False,
    "discounts": "unset pending margin and commercial review",
    "customer_assignments": "existing assigned segments only; unresolved assignments remain separate",
    "primary_metric": "proposed incremental contribution margin per randomized eligible customer",
}
(output_dir / "campaign_design_settings.json").write_text(json.dumps(settings, indent=2))
print(f"Exported {len(plan)} draft campaign hypotheses; no campaigns launched.")
review_fields = ["campaign_id", "cluster", "campaign_name", "analytical_decision",
    "holdout_review_gate", "segment_fit_status", "segment_fit_rationale", "decision_rationale",
    "mechanic_rationale", "segment_specificity_note", "review_status", "launch_approved"]
plan[review_fields].to_csv(output_dir / "campaign_analytical_review.csv", index=False)
settings.update({
    "revision": "part_c_analytical_review_v2",
    "decisions": "qualitative post-hoc review; not a numerical ranking or optimal selection",
    "profile_validation": "not independently re-audited in Part C",
    "status_separation": "descriptive evidence gate, analytical decision, commercial review, launch approval",
})
(output_dir / "campaign_design_settings.json").write_text(json.dumps(settings, indent=2))

Exported 6 draft campaign hypotheses; no campaigns launched.


834

## 8. Conclusion and report interpretation

The six proposals differ in evidence strength and commercial plausibility. Passing the descriptive holdout check does not give them equal priority. The table below is computed from the current Part B export rather than fixed figures copied from previous runs.

- C0-01: retain a modest meal-companion hypothesis, without claiming segment exclusivity.
- C1-01: keep as an exploratory alternative; food pairing does not explain the electronics/gaming profile.
- C2-01: retain for exact product and recipe compatibility review, not automatic coupon approval.
- C3-01: deprioritize until a credible cross-aisle use case is established.
- C4-01: retain a recipe-led hypothesis while stating the outside-segment comparison explicitly.
- C5-01: retain for limited exploratory review; small supporting volume warrants caution.

These are post-hoc judgments informed by the already-inspected holdout. They are not independently confirmed campaign choices. No incremental-sales effect, optimal discount, superiority over other segments, or campaign ROI has been established.

Before submission, finish the profile-based rationale in the executive report, preserve uncertainty and rejected ideas, and complete the separate all-customer assignment deliverable. No further mining is required to finish this campaign-design stage.